**Linear SVM**

Install libraire

In [73]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix



In [51]:
# chargement du dataset
df = pd.read_csv("../data/dataset_avis.csv")
df.head()

,comment_id,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,qualité produit,service livraison,service client
0,2,"Vente Lacoste Honteuse , article erroné , arti...",1,2021-06-19 00:00:00+00:00,Vanessa L,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,"vente lacoste honteuse , article erroné , arti...",1,0,0
1,6,Annulation de commande après 2 mois d ’ attent...,1,2021-06-18 00:00:00+00:00,aurore regnier,"Bonjour Aurore , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,annulation de commande après 2 mois d ’ attent...,0,0,1
2,8,Extrêmement deçu pour mes achats lors la vente...,1,2021-06-18 00:00:00+00:00,Ayna,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,extrêmement deçu pour mes achats lors la vente...,1,0,0
3,9,S'il y'avait une option : ne pas mettre d'étoi...,1,2021-06-18 00:00:00+00:00,linda Ng,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,s'il y'avait une option : ne pas mettre d'étoi...,1,0,0
4,10,ARNAQUE J ’ ai acheté une combinaison blanche ...,1,2021-06-18 00:00:00+00:00,Sarah,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,arnaque j ’ ai acheté une combinaison blanche ...,1,0,0


In [37]:
# colonnes cibles(multilabel)
labels = ["qualité produit", "service livraison", "service client"]


In [77]:
# Données texte (features)
X = df["clean_comment"]



In [76]:
# Données cibles
y = df[labels].values


In [47]:
#TF-IDF vectorization

tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

X_tfidf = tfidf.fit_transform(X)

In [ ]:
#Séparation Train / Test
# Nous séparons le dataset en ensembles d'entraînement et de test (80/20).
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Modèle SVM Multilabel
# nous entraînons un classifieur LinearSVC dans un schéma OneVsRest pour gérer les labels multiples.
# LinearSVC as base model
svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000
)

# multilable avec OneVsRest
clf = OneVsRestClassifier(svm_model)

In [ ]:
#Validation croisée (F1 micro)
#Évaluation du modèle sur 5 folds avec le F1-score micro.
scorer = make_scorer(f1_score, average="micro")

cv_scores = cross_val_score(
    clf,
    X_tfidf,
    y,
    cv=5,
    scoring=scorer,
    n_jobs=-1
)

print("F1 micro per fold :", cv_scores)
print("F1 micro average  :", cv_scores.mean())
print("Standard deviation :", cv_scores.std())

F1 micro per fold : [0.64876678 0.76193922 0.7558098  0.76108108 0.60930802]
F1 micro average  : 0.7073809798006359
Standard deviation : 0.06520672032258247


Entraînement et évaluation sur le test set

In [ ]:
# Train on train set
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)



# Scores per label
for i, col in enumerate(labels):
    acc = np.mean(y_test[:, i] == y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# # Scores globaux
f1_micro = f1_score(y_test, y_pred, average="micro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"\nF1 micro    : {f1_micro:.4f}")
print(f"F1 weighted : {f1_weighted:.4f}")
#  Évaluation sur le jeu de test
print(classification_report(y_test, y_pred, target_names=labels))


Label 'qualité produit': Accuracy = 0.744, F1-score = 0.774
Label 'service livraison': Accuracy = 0.774, F1-score = 0.716
Label 'service client': Accuracy = 0.828, F1-score = 0.463

F1 micro    : 0.7094
F1 weighted : 0.7085
                   precision    recall  f1-score   support

  qualité produit       0.75      0.80      0.77       717
service livraison       0.75      0.68      0.72       547
   service client       0.46      0.46      0.46       209

        micro avg       0.71      0.71      0.71      1473
        macro avg       0.65      0.65      0.65      1473
     weighted avg       0.71      0.71      0.71      1473
      samples avg       0.70      0.73      0.70      1473



c:\Users\vires\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Matrices de confusion

In [59]:

labels = ["qualité produit", "service livraison", "service client"]
for i, label in enumerate(labels):
    cm = confusion_matrix(y_test[:, i], y_pred[:, i])

    print(f"Confusion matrix for {label}:")
   # print(cm)
    cm_df = pd.DataFrame(cm,index = ["Vrai 0","Vrai 1"],columns = ["Prédit 0","Prédit 1"])
    print(cm_df)

Confusion matrix for qualité produit:
        Prédit 0  Prédit 1
Vrai 0       397       193
Vrai 1       142       575
Confusion matrix for service livraison:
        Prédit 0  Prédit 1
Vrai 0       638       122
Vrai 1       174       373
Confusion matrix for service client:
        Prédit 0  Prédit 1
Vrai 0       985       113
Vrai 1       112        97


Évaluation sur 100_avis_annote.csv

In [ ]:
#Évaluation sur 100_avis_annote.csv
df_avis = pd.read_csv("../data/test_dataset/100_avis_annote.csv", sep=";")
df_avis.head()

,comment_id,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,qualité produit,service livraison,service client
0,9720,Très bonne expérience . Ras,5,28/06/2020,NaN,NaN,TrustedShop,ShowRoom,NaN,NaN,NaN,NaN,très bonne expérience . ras,0,0,0
1,8772,Bonjour je n ai toujours pas reçu ma commande,1,06/07/2020,NaN,NaN,TrustedShop,ShowRoom,NaN,NaN,NaN,NaN,bonjour je n ai toujours pas reçu ma commande,0,1,0
2,11565,Commender au mois de mai reçu au mois de juin ...,3,13/06/2020,NaN,NaN,TrustedShop,ShowRoom,NaN,NaN,NaN,NaN,commender au mois de mai reçu au mois de juin ...,0,1,0
3,4115,Très satisfaite de ma commande,5,14/11/2020,Véronique H .,"Bonjour , Merci pour votre gentil message , no...",TrustedShop,ShowRoom,BUCHELAY,NaN,02/11/2020,12.0,très satisfaite de ma commande,0,0,0
4,6363,Tout s'est très bien passé,5,07/08/2020,Mireille C .,NaN,TrustedShop,ShowRoom,Manosque,NaN,NaN,NaN,tout s'est très bien passé,0,0,0


In [ ]:
# TF-IDF
X_avis_tfidf = tfidf.transform(df_avis['clean_comment'])
y_avis_true = df_avis[["qualité produit", "service livraison", "service client"]].values

In [ ]:
# Prédiction
y_avis_pred = clf.predict(X_avis_tfidf)

In [ ]:
# Scores per label
for i, col in enumerate(labels):
    acc = np.mean(y_avis_true[:, i] == y_avis_pred[:, i])
    f1 = f1_score(y_avis_true[:, i],y_avis_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores globaux
f1_micro = f1_score(y_avis_true, y_avis_pred, average="micro")
f1_weighted = f1_score(y_avis_true, y_avis_pred, average="weighted")

print(f"\nF1 micro    : {f1_micro:.4f}")
print(f"F1 weighted : {f1_weighted:.4f}")

Label 'qualité produit': Accuracy = 0.700, F1-score = 0.667
Label 'service livraison': Accuracy = 0.690, F1-score = 0.392
Label 'service client': Accuracy = 0.700, F1-score = 0.167

F1 micro    : 0.4859
F1 weighted : 0.4502


In [ ]:
# Classification report
print("SVM Results on the 100 Manual Labels:")
print(classification_report(y_avis_true, y_avis_pred, target_names=labels))

SVM Results on the 100 Manual Labels:
                   precision    recall  f1-score   support

  qualité produit       0.61      0.73      0.67        41
service livraison       0.33      0.48      0.39        21
   service client       0.33      0.11      0.17        27

        micro avg       0.49      0.48      0.49        89
        macro avg       0.43      0.44      0.41        89
     weighted avg       0.46      0.48      0.45        89
      samples avg       0.41      0.36      0.37        89



c:\Users\vires\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\vires\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\vires\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Matrices de confusion

for i, label in enumerate(labels):
    cm = confusion_matrix(y_avis_true[:, i], y_avis_pred[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"\nMatrice de confusion — {label}")
    display(cm_df)


Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,40,19
Vrai 1,11,30



Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,59,20
Vrai 1,11,10



Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,67,6
Vrai 1,24,3
